In [ ]:
%load_ext autoreload
%autoreload 2

import geopandas as gpd

import src.data_filepaths as data_filepaths
import src.utils.loaders as loaders
import src.utils.vis.colors_utils as colors_utils
from src.utils.loaders import process_loaded_building_df
from src.utils.vis.map import MapExplorer

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
building_gdf = gpd.read_parquet(data_filepaths.OVERTURE_BUILDING)
# b_gdf = building_gdf.to_crs("EPSG:3414")
building_part_gdf = gpd.read_parquet(data_filepaths.OVERTURE_BUILDING_PART)
# bp_gdf = building_part_gdf.to_crs("EPSG:3414")

print(building_gdf.dtypes)
print("")
print(building_part_gdf.dtypes)

print(building_gdf.columns, building_part_gdf.columns)

id                             str
names                       object
sources                     object
level                      float64
height                     float64
min_height                 float64
is_underground                bool
num_floors                 float64
num_floors_underground     float64
min_floor                  float64
subtype                        str
class                          str
facade_color                   str
facade_material                str
roof_material                  str
roof_shape                     str
roof_direction             float64
roof_orientation               str
roof_color                     str
roof_height                float64
geometry                  geometry
has_parts                     bool
version                      int32
bbox                        object
theme                          str
type                           str
dtype: object

id                             str
names                       object
heigh

In [8]:
out = process_loaded_building_df(building_gdf)

print("\nSUBTYPE")
print(out["subtype"].value_counts(dropna=False).head(100))
print(out["subtype"].unique())
print("\nCLASS")
print(out["class"].value_counts(dropna=False).head(100))
print(out["class"].unique())


SUBTYPE
subtype
NaN               180933
residential        43501
commercial          3822
industrial          3035
outbuilding         1484
civic               1441
transportation      1290
education            787
religious            598
service              573
entertainment        167
medical              116
agricultural          85
military               1
Name: count, dtype: int64
<ArrowStringArray>
[             nan,     'industrial',     'commercial', 'transportation',
    'residential',          'civic',    'outbuilding',        'service',
      'education',  'entertainment',      'religious',        'medical',
   'agricultural',       'military']
Length: 14, dtype: str

CLASS
class
NaN                 182636
residential          17220
house                16102
apartments            3415
industrial            3034
                     ...  
bunker                   1
bridge_structure         1
kiosk                    1
monastery                1
supermarket              1

In [9]:
def test_configurable():
    mapper = MapExplorer(
        building_gdf=process_loaded_building_df(building_gdf),
        building_part_gdf=process_loaded_building_df(building_part_gdf),
        # enable_debug_mode=True,
    )

    mapper.build_configurable(
        viewable_columns=[
            "height_calc",
            "subtype",
            "class",
            "num_floors",
            "area_m2",
            "src_dataset_label",
        ],
        height_cmap="jet",
        show_histogram=False, # it looks bad atm
        title="Test Title",
    )

# test_configurable()

In [10]:
# singleprop test
def test_singleprop():
    mapper = MapExplorer(
        building_gdf=process_loaded_building_df(building_gdf),
        building_part_gdf=process_loaded_building_df(building_part_gdf),
        enable_debug_mode=True,
    )

    cmap = colors_utils.CategoricalColorMap(
        cat_colors=[
            colors_utils.get_color_by_name("blue"),
            colors_utils.get_color_by_name("red"),
        ],
        cat_labels=[True, False],
    )

    return mapper.build_singleprop(
        column="is_underground",
        categorical_cmap=cmap,
        title="Singleprop Test Title",
    )

# test_singleprop()

## Masterplan

### Initial

In [11]:
landuse_gdf = gpd.read_file(data_filepaths.MASTERPLAN_LANDUSE)

print(landuse_gdf.columns)
print(landuse_gdf.dtypes)
print(len(landuse_gdf))
landuse_gdf.head()

Index(['OBJECTID', 'LU_DESC', 'LU_TEXT', 'GPR', 'WHI_Q_MX', 'GPR_B_MN',
       'INC_CRC', 'FMEL_UPD_D', 'SHAPE.AREA', 'SHAPE.LEN', 'geometry'],
      dtype='str')
OBJECTID         int32
LU_DESC            str
LU_TEXT            str
GPR                str
WHI_Q_MX       float64
GPR_B_MN       float64
INC_CRC            str
FMEL_UPD_D         str
SHAPE.AREA     float64
SHAPE.LEN      float64
geometry      geometry
dtype: object
113394


,OBJECTID,LU_DESC,LU_TEXT,GPR,WHI_Q_MX,GPR_B_MN,INC_CRC,FMEL_UPD_D,SHAPE.AREA,SHAPE.LEN,geometry
0,692393,RESIDENTIAL,NaN,LND,NaN,NaN,9FFCB6622FA68945,20251229172155,14.660606,49.020738,"POLYGON ((103.88929 1.30137, 103.88929 1.30138..."
1,692394,PARK,NaN,EVA,NaN,NaN,DB7CFCDC2489D32D,20251229172153,2142.094320,271.971782,"POLYGON ((103.91037 1.32037, 103.91041 1.32043..."
2,692395,PARK,NaN,EVA,NaN,NaN,56E4AC91F39ECD54,20251229172153,16506.639461,535.609052,"POLYGON ((103.91492 1.32829, 103.91484 1.32837..."
3,692396,RESIDENTIAL,NaN,LND,NaN,NaN,5274C36C4F106B3B,20251229172153,201.080143,56.827867,"POLYGON ((103.94437 1.33387, 103.94429 1.33398..."
4,692397,RESIDENTIAL,NaN,1.4,NaN,NaN,65A8187DD54FA5A2,20251229172153,347.066144,94.675301,"POLYGON ((103.90724 1.32378, 103.90724 1.32386..."


In [12]:
def test_landuse_simple():
    mapper = MapExplorer(
        building_gdf=process_loaded_building_df(building_gdf),
        building_part_gdf=process_loaded_building_df(building_part_gdf),
        # enable_debug_mode=True,
    )


    mapper.build_configurable_landuse(
        landuse_gdf, viewable_columns=["LU_DESC", "LU_TEXT", "WHI_Q_MX", "GPR_B_MN", "GPR"]
    )

### Using the Util

In [ ]:
landuse_gdf = loaders.load_masterplan_landuse_geojson_to_gdf(data_filepaths.MASTERPLAN_LANDUSE)
# landuse_gdf.head(12)

: 

In [ ]:
# mapper.build_configurable_landuse(
#     landuse_gdf, viewable_columns=["landuse", "gpr", "is_landed", "is_sdp"]
# )

In [ ]:
# landuse_gdf["color"].value_counts()

In [ ]:
# singleprop test
def test_landuse_buildings():
    mapper = MapExplorer(
        building_gdf=process_loaded_building_df(building_gdf),
        building_part_gdf=process_loaded_building_df(building_part_gdf),
        # enable_debug_mode=True,
    )

    cmap = colors_utils.CategoricalColorMap(
        cat_colors=[
            colors_utils.get_color_by_name("blue"),
            colors_utils.get_color_by_name("red"),
        ],
        cat_labels=[True, False],
    )

    return mapper.build_singleprop(
        column="class",
        # categorical_cmap=cmap,
        colored_landuse_gdf=landuse_gdf,
        title="Landuse + Buildings Title",
    )


test_landuse_buildings()

## OSM Landuse

# Janky Todo list

Add 

improve UI configurability

adding Masterplan support

* basically, in build, you can supply your own base layer for display, with colour/colormap
* also add custom titles :)

In [ ]:
# process_loaded_building_df(building_gdf)["num_floors"].value_counts()